# FireGrounder V3 - Hướng dẫn chạy trong Jupyter Notebook

**Dự án**: Dự đoán **điểm gốc lửa tiếp xúc với mặt sàn** (fire-base contact point) từ ảnh RGB/video/webcam.

**Output**: `[p_fire, x_norm, y_norm]` — xác suất có lửa + tọa độ chuẩn hóa của chân nền ngọn lửa.

## Prerequisites
1. Mở terminal/cmd, kích hoạt môi trường conda:
   ```cmd
   conda activate firegrounder
   ```
2. Chạy notebook này trong môi trường đã kích hoạt.

## Dataset
- `fire_ground_dataset/dataset_labels.json`: 6,500 mẫu (4,509 có lửa, 1,991 không lửa)
- **Bạn cần tải về bộ ảnh HomeFire** và đặt đường dẫn trong biến môi trường `FIRE_IMAGE_ROOT`
- Dataset chỉ chứa labels, **không có ảnh** trong thư mục này.

## Ô 1: Kiểm tra môi trường

Kiểm tra Python, PyTorch, CUDA và các thư viện cần thiết.

In [ ]:
import torch
import sys
import os

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

import timm
print(f"timm: {timm.__version__}")

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

print(f"OpenCV: {cv2.__version__}")
print(f"NumPy: {np.__version__}")
print("✅ Môi trường đã sẵn sàng!

## Ô 2: Import FireGrounder V3 và khởi tạo model

Tải model V3 với MobileNetV4 backbone + FPN-lite heatmap head.

In [ ]:
import sys
sys.path.insert(0, '.')  # Đảm bảo thư mục project có trong PATH

from firegrounder_v3 import (
    FireGrounderV3,
    FireLossV3,
    load_checkpoint_v3,
    val_transform_v3,
    IMG_SIZE,
    HEATMAP_SIZE,
    BACKBONE_NAME,
    FIRE_THRESHOLD,
)

# Chọn thiết bị
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Sử dụng thiết bị: {device}")

# Khởi tạo model
model = FireGrounderV3(
    backbone_name=BACKBONE_NAME,
    pretrained=True,  # Tải ImageNet pretrained weights
    img_size=IMG_SIZE,
    heatmap_size=HEATMAP_SIZE,
    softargmax_temperature=0.07
).to(device)

model.eval()  # Chuyển sang chế độ inference
print("✅ Model đã sẵn sàng!")

## Ô 3: Chạy Smoke Test

Kiểm tra model với dữ liệu ngẫu nhiên để đảm bảo không lỗi.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Tạo tensor ngẫu nhiên
dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device)

with torch.no_grad():
    outputs = model(dummy_input, return_dict=True)

print(f"Prediction shape: {outputs['pred'].shape}")  # (2, 3)
print(f"Heatmap shape: {outputs['heatmap_logits'].shape}")  # (2, 1, 64, 64)
print(f"p_fire: {outputs['p_fire'].cpu().numpy()}")
print(f"xy coords: {outputs['xy'].cpu().numpy()}")
print(f"Heatmap logits range: [{outputs['heatmap_logits'].min().item():.2f}, {outputs['heatmap_logits'].max().item():.2f}]")

# Kiểm tra giá trị hợp lệ
assert outputs['pred'].shape == (2, 3)
assert outputs['heatmap_logits'].shape == (2, 1, 64, 64)
assert torch.isfinite(outputs['pred']).all()
print("✅ Smoke test passed!")

## Ô 4: Training Pipeline (notebook4_v3_fpn.py)

**Yêu cầu**: Bạn cần tải bộ ảnh HomeFire dataset về máy và đặt đường dẫn đúng.

### Cấu hình thông qua biến môi trường:
```python
os.environ['FIRE_RUN_TRAIN'] = '1'                    # Bật training
os.environ['FIRE_JSON_PATH'] = './fire_ground_dataset/dataset_labels.json'
os.environ['FIRE_IMAGE_ROOT'] = 'D:/duong/dan/toi/home-fire-dataset'  # THAY ĐỔI!
os.environ['FIRE_OUTPUT_DIR'] = './v3_outputs'  # Nơi lưu checkpoint
```

Sau khi thiết lập, uncomment dòng `%run notebook4_v3_fpn.py` dưới đây.

In [ ]:
import os

# ⚠️ THIẾT LẬP BIẾN MÔI TRƯỜNG TRƯỚC KHI CHẠY TRAINING
# os.environ['FIRE_RUN_TRAIN'] = '1'
# os.environ['FIRE_JSON_PATH'] = './fire_ground_dataset/dataset_labels.json'
# os.environ['FIRE_IMAGE_ROOT'] = 'D:/path/to/your/home-fire-dataset'  # THAY ĐỔI!
# os.environ['FIRE_OUTPUT_DIR'] = './v3_outputs'

# %run notebook4_v3_fpn.py

print("⚠️ Để chạy training:")
print("1. Tải home-fire-dataset về máy")
print("2. Đặt đúng FIRE_IMAGE_ROOT trong ô code trên")
print("3. Uncomment dòng %run notebook4_v3_fpn.py")
print("\n📝 Checkpoint sẽ lưu tại: ./v3_outputs/best_v3_fpn.pth

## Ô 5: Evaluation

Đánh giá model trên validation set. Cần có checkpoint V3.

In [ ]:
import os

os.environ['FIRE_RUN_EVAL'] = '1'
os.environ['FIRE_RUN_PREVIEW'] = '1'
os.environ['FIRE_JSON_PATH'] = './fire_ground_dataset/dataset_labels.json'
os.environ['FIRE_IMAGE_ROOT'] = 'D:/path/to/your/home-fire-dataset'
os.environ['FIRE_CHECKPOINT_PATH'] = './v3_outputs/best_v3_fpn.pth'

# %run notebook4_v3_fpn.py

print("⚠️ Để chạy evaluation:")
print("1. Cần có checkpoint V3 tại ./v3_outputs/best_v3_fpn.pth")
print("2. Uncomment dòng %run notebook4_v3_fpn.py")
print("\n📝 Kết quả sẽ lưu tại:")
print("  - v3_outputs/v3_eval_metrics.json")
print("  - v3_outputs/v3_val_preview.png

## Ô 6: Inference trên ảnh đơn lẻ

Sử dụng model đã được huấn luyện để dự đoán fire base point từ ảnh.

In [ ]:
from inference_v3 import predict, draw_fire_base
from PIL import Image
import matplotlib.pyplot as plt

# Cấu hình đường dẫn
checkpoint_path = './v3_outputs/best_v3_fpn.pth'  # Hoặc checkpoint của bạn
image_path = './fire.jpg'  # Thay đổi đường dẫn ảnh của bạn

try:
    # Load model
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = load_checkpoint_v3(checkpoint_path, device)
    
    # Dự đoán
    image = Image.open(image_path).convert('RGB')
    p_fire, x_norm, y_norm = predict(model, image, device)
    
    print("Kết quả dự đoán:")
    print(f"  p_fire = {p_fire:.4f} ({'🔥 FIRE' if p_fire >= FIRE_THRESHOLD else '✅ NO FIRE'})")
    print(f"  x_norm = {x_norm:.4f}")
    print(f"  y_norm = {y_norm:.4f}")
    print(f"  Pixel  : x={x_norm * image.width:.1f}, y={y_norm * image.height:.1f} (trên ảnh gốc)")
    
    # Vẽ kết quả lên ảnh
    import cv2
    import numpy as np
    frame = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    result = draw_fire_base(frame, p_fire, x_norm, y_norm, threshold=FIRE_THRESHOLD)
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(result_rgb)
    plt.title(f"FireBase Detection | Confidence: {p_fire*100:.1f}%")
    plt.axis('off')
    plt.show()
    
except FileNotFoundError as e:
    print(f"⚠️ Lỗi: {e}")
    print("Hãy chắc chắn:")
    print(f"  1. Checkpoint tồn tại tại: {checkpoint_path}")
    print(f"  2. Ảnh tồn tại tại: {image_path}")
    print("\n💡 Chạy Ô 4 để training trước khi inference, hoặc cập nhật đường dẫn checkpoint.

## Ô 7: Inference trên video/webcam

### Video:
```python
import os
os.environ['FIRE_CHECKPOINT_PATH'] = './v3_outputs/best_v3_fpn.pth'

# Run with command line args:
# python inference_v3.py --mode video --input fire.mp4 --checkpoint ./v3_outputs/best_v3_fpn.pth --output fire_v3_result.mp4 --device auto
```

### Webcam:
```python
# python inference_v3.py --mode webcam --checkpoint ./v3_outputs/best_v3_fpn.pth --device auto --show
```

## Tóm tắt các file chính:

| File | Mô tả | Cách chạy |
|------|-------|-----------|
| `firegrounder_v3.py` | Model V3 (MobileNetV4 + FPN-lite + heatmap) | Import trong notebook |
| `notebook4_v3_fpn.py` | Pipeline train/eval | `%run notebook4_v3_fpn.py` |
| `inference_v3.py` | Inference CLI | `python inference_v3.py --mode image --input fire.jpg` |
| `fire_ground_dataset/dataset_labels.json` | Dataset nhãn (6,500 mẫu) | Tự động phát hiện |

## Lưu ý quan trọng:
1. **Dataset chỉ có labels**, bạn cần tự tải ảnh HomeFire về
2. Đặt đúng `FIRE_IMAGE_ROOT` trước khi train/eval
3. Dùng checkpoint V3 với `inference_v3.py`, checkpoint V2 với `inference.py`
4. CUDA đã sẵn sàng trên máy bạn (NVIDIA 3060) ✅